<a href="https://colab.research.google.com/github/vruddhis/semanticshift/blob/main/table2_disruption_drift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# TODO:TABLE 3 AND TABLE 5
# get files
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
import pandas as pd
import os
import numpy as np
from sklearn.cluster import KMeans, MiniBatchKMeans
from scipy.stats import entropy

In [18]:
informal_embed_path = '/content/drive/MyDrive/semantic_shift/datasets/embeddings/informal_embeds.csv'
formal_embed_path = '/content/drive/MyDrive/semantic_shift/datasets/embeddings/formal_embeds.csv'

df_inf = pd.read_csv(informal_embed_path)
df_inf.head()

df_f= pd.read_csv(formal_embed_path)
df_f.head()

,word,event,register,period,sentence,timestamp,embedding
0,tag,social-media,formal,before,The service is backed by some well-lnown chara...,2008-06-18T17:49:07Z,"[-0.10919109731912613, 0.28668826818466187, 0...."
1,tag,social-media,formal,before,Here's the next part: if they need to fill in ...,2008-09-26T12:16:17Z,"[0.05975320562720299, 0.1855640560388565, -0.1..."
2,tag,social-media,formal,before,The £199 price tag didn't bother him - the Rea...,2008-09-04T09:23:00Z,"[-0.08915523439645767, 0.341594934463501, -0.1..."
3,tag,social-media,formal,before,"Or for those watching the pennies, there's the...",2008-11-18T00:01:00Z,"[0.09958921372890472, 0.007577353622764349, 0...."
4,tag,social-media,formal,before,The intro or article abstract Many publication...,2007-11-19T07:44:17Z,"[0.08138493448495865, 0.03273022547364235, 0.7..."


In [ ]:
def process_data(file_path, register_label):
    ALPHA = 0.5
    BETA = 0.5

    df = pd.read_csv(file_path).reset_index(drop=True)

    def parse_emb(x):
        return np.fromstring(x.strip('[]'), sep=',')

    print("Parsing embeds:")
    embeddings_raw = np.stack(df['embedding'].apply(parse_emb).values)

    def get_entropy(emb_subset, n_clusters=10):
        if len(emb_subset) < n_clusters: return 0
        mbk = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, n_init="auto")
        clusters = mbk.fit_predict(emb_subset)
        _, counts = np.unique(clusters, return_counts=True)
        return entropy(counts / counts.sum())

    results = []
    words = df['word'].unique()

    for word in words:
        word_data = df[df['word'] == word]

        idx_before = word_data[word_data['period'] == 'before'].index
        idx_event  = word_data[word_data['period'] == 'event'].index
        idx_after  = word_data[word_data['period'] == 'after'].index

        # metadata
        event_id = word_data['event'].iloc[0] if 'event' in word_data.columns else "N/A"

        if len(idx_before) > 0:
        #entropy calcs
            h_before = get_entropy(embeddings_raw[idx_before])
            
        else:
            h_before = 0.1
            
        if len(idx_after) > 0:
            h_after = get_entropy(embeddings_raw[idx_after])
            
        else:
            h_after = 0.1

        #disruption
        before_embs = embeddings_raw[idx_before]
        chunks = np.array_split(before_embs, 5)
        h_baseline_samples = [get_entropy(c) for c in chunks if len(c) > 5]

        sigma_base = np.std(h_baseline_samples) if len(h_baseline_samples) > 1 else 0.01
        mu_base = np.mean(h_baseline_samples) if len(h_baseline_samples) > 0 else 0.1

        if len(idx_event) == 0:
            h_event = mu_base
        else:
            h_event = get_entropy(embeddings_raw[idx_event])

        # Scores final
        drift = (h_after - h_before) / 3.32
        disruption = (sigma_base / (mu_base + 1e-9)) * 10
        h_final = (ALPHA * drift) + (BETA * disruption)

        base_info ={
            'target word': word,
            'event id': event_id,
            'register': register_label,
            'entropy shift': h_final,
            'drift score': drift,
            'disruption score': disruption
        }

        results.append(base_info)

        # for s_type in ['baseline', 'disruption', 'short term drift', 'long term drift']:
        #     results.append({**base_info, 'shift type': s_type})

    final_meme_table = pd.DataFrame(results)

    return final_meme_table[[
        'target word', 'event id', 'register', 'entropy shift',
         'drift score', 'disruption score'
    ]]

In [20]:
#both registers
informal_results = process_data(informal_embed_path, 'Informal')
formal_results = process_data(formal_embed_path, 'Formal')

print("Disruption informal")
display(informal_results.sort_values(by='disruption score', ascending=False).head(5))

print("Disruption formal")
display(formal_results.sort_values(by='disruption score', ascending=False).head(5))

print("Drift informal")
display(informal_results.sort_values(by='drift score', ascending=False).head(5))

print("Drift formal")
display(formal_results.sort_values(by='drift score', ascending=False).head(5))

informal_results.to_csv('/content/drive/MyDrive/semantic_shift/colab_outputs/informal_table2.csv', index=False)
formal_results.to_csv('/content/drive/MyDrive/semantic_shift/colab_outputs/formal_table2.csv', index=False)


# IMP COMMENTS: MIGHT NEED LATER


# final_comparison.to_csv('path', index=False)

# comparison_inf = informal_results[informal_results['shift type'] == 'long term drift']
# comparison_for = formal_results[formal_results['shift type'] == 'long term drift']

# # Merge them into one table for final analysis
# final_comparison = pd.merge(
#     comparison_inf,
#     comparison_for,
#     on='target word',
#     suffixes=('_inf', '_for')
# )

# Calculate Register Delta for both metrics
# final_comparison['drift_delta'] = abs(final_comparison['drift_score_inf'] - final_comparison['drift_score_for'])
# final_comparison['disruption_delta'] = abs(final_comparison['disruption_score_inf'] - final_comparison['disruption_score_for'])

# disruptive score
# display(final_comparison.sort_values(by='disruption_score_inf', ascending=False).head(5))

# drift score
# display(final_comparison.sort_values(by='drift_delta', ascending=False).head(5))

Parsing embeds:
Parsing embeds:
Disruption informal


,target word,event id,register,entropy shift,drift score,disruption score
1,binge,streaming,Informal,0.261245,0.016838,0.505652
7,catfish,catfish-show,Informal,0.187178,-0.116212,0.490568
5,algorithm,ai-automation,Informal,0.156432,-0.010938,0.323801
0,creator,social-media,Informal,0.148943,-0.000539,0.298425
2,cancel,me-too,Informal,0.153478,0.012473,0.294484


Disruption formal


,target word,event id,register,entropy shift,drift score,disruption score
7,tweet,social-media,Formal,0.812329,0.143816,1.480841
8,remote,covid-19,Formal,0.466638,0.022657,0.910619
3,prompt,chatgpt-launch,Formal,0.355369,-0.042348,0.753087
6,spoiler,streaming,Formal,0.362463,-0.005656,0.730582
10,stream,streaming,Formal,0.247744,0.029624,0.465865


Drift informal


,target word,event id,register,entropy shift,drift score,disruption score
1,binge,streaming,Informal,0.261245,0.016838,0.505652
2,cancel,me-too,Informal,0.153478,0.012473,0.294484
3,beta,fanfic,Informal,0.086611,0.002161,0.171061
0,creator,social-media,Informal,0.148943,-0.000539,0.298425
5,algorithm,ai-automation,Informal,0.156432,-0.010938,0.323801


Drift formal


,target word,event id,register,entropy shift,drift score,disruption score
7,tweet,social-media,Formal,0.812329,0.143816,1.480841
10,stream,streaming,Formal,0.247744,0.029624,0.465865
1,drop,nft,Formal,0.232658,0.027649,0.437667
4,variant,covid-19,Formal,0.191793,0.026766,0.356819
8,remote,covid-19,Formal,0.466638,0.022657,0.910619
